In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

RAW_PATH = "../data/Nassau_Candy_Distributor.csv"
OUT_PATH = "../outputs/cleaned_data.csv"

LEAD_TIME_RANGES = {
    "Same Day": (0, 1),
    "First Class": (1, 3),
    "Second Class": (3, 5),
    "Standard Class": (5, 8),
}

In [3]:
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(10194, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [4]:
def simulate_lead_time(ship_mode_series):
    lead_times = np.zeros(len(ship_mode_series))

    for mode, (lo, hi) in LEAD_TIME_RANGES.items():
        mask = (ship_mode_series == mode).values
        n = mask.sum()
        if n == 0:
            continue
        mode_point = lo + (hi - lo) * 0.3
        sampled = np.random.triangular(lo, mode_point, hi, n)
        lead_times[mask] = sampled

    noise = np.random.normal(0, 0.4, len(lead_times))
    lead_times = np.clip(lead_times + noise, 0, 10)
    return np.round(lead_times, 2)

In [5]:
df["Lead Time"] = simulate_lead_time(df["Ship Mode"])
df.groupby("Ship Mode")["Lead Time"].describe()

,count,mean,std,min,25%,50%,75%,max
Ship Mode,,,,,,,,
First Class,1548.0,1.866796,0.566082,0.00,1.47,1.83,2.2425,3.89
Same Day,547.0,0.461993,0.380977,0.00,0.12,0.43,0.7100,1.94
Second Class,1979.0,3.875336,0.583682,2.19,3.48,3.86,4.2800,5.87
Standard Class,6120.0,6.284248,0.736778,4.11,5.76,6.23,6.7900,8.74


In [6]:
import sys
sys.path.append(".")  # lets the notebook import from files in the same folder

from state_coords import STATE_COORDS

factories = pd.read_csv("../data/factories.csv")
pf_map = pd.read_csv("../data/product_factory_map.csv")

factories

,Factory,Latitude,Longitude
0,Lot's O' Nuts,32.881893,-111.768036
1,Wicked Choccy's,32.076176,-81.088371
2,Sugar Shack,48.119140,-96.181150
3,Secret Factory,41.446333,-90.565487
4,The Other Factory,35.117500,-89.971107


In [7]:
def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

In [8]:
df = df.merge(pf_map[["Product Name", "Factory"]], on="Product Name", how="left")
df = df.merge(factories, on="Factory", how="left")
df = df.rename(columns={"Latitude": "Factory_Lat", "Longitude": "Factory_Lon"})

df[["Product Name", "Factory", "Factory_Lat", "Factory_Lon"]].head()

,Product Name,Factory,Factory_Lat,Factory_Lon
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,32.076176,-81.088371
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,32.076176,-81.088371
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,32.881893,-111.768036
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,32.881893,-111.768036
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,32.076176,-81.088371


In [9]:
missing_factory = df["Factory"].isna().sum()
print(f"Rows with no factory mapping: {missing_factory}")

coords = df["State/Province"].map(STATE_COORDS)
missing_coords = coords.isna().sum()
print(f"Rows with no state coordinate: {missing_coords}")

df["Dest_Lat"] = coords.apply(lambda x: x[0])
df["Dest_Lon"] = coords.apply(lambda x: x[1])

Rows with no factory mapping: 0
Rows with no state coordinate: 0


In [10]:
df["Shipping_Distance_KM"] = haversine(
    df["Factory_Lat"], df["Factory_Lon"], df["Dest_Lat"], df["Dest_Lon"]
)

df[["Product Name", "Factory", "Region", "Shipping_Distance_KM"]].sample(10)

,Product Name,Factory,Region,Shipping_Distance_KM
6134,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Interior,2402.075101
7990,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Atlantic,3174.127944
1808,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Interior,2188.313441
4008,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Pacific,2371.477446
3169,Lickable Wallpaper,Secret Factory,Pacific,2579.335329
9458,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Gulf,2318.506897
381,Wonka Bar - Milk Chocolate,Wicked Choccy's,Gulf,686.586104
7175,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Pacific,2371.477446
6747,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Atlantic,3377.542037
9769,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Pacific,809.097829


In [13]:
for col in ["Sales", "Cost", "Gross Profit"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    before = len(df)
    df = df[(df[col] >= lo) & (df[col] <= hi)]
    removed = before - len(df)
    print(f"Removed {removed} outlier rows on {col}")

Removed 0 outlier rows on Sales
Removed 0 outlier rows on Cost
Removed 0 outlier rows on Gross Profit


In [14]:
df["Profit_Margin"] = df["Gross Profit"] / df["Sales"]
df["Cost_Per_Unit"] = df["Cost"] / df["Units"]

df[["Sales", "Gross Profit", "Profit_Margin", "Cost", "Units", "Cost_Per_Unit"]].describe()

,Sales,Gross Profit,Profit_Margin,Cost,Units,Cost_Per_Unit
count,10045.000000,10045.000000,10045.000000,10045.000000,10045.000000,10045.000000
mean,13.103870,8.783803,0.667765,4.320067,3.742260,1.159944
std,7.590325,5.184803,0.060332,2.538035,2.140051,0.265880
min,1.250000,0.250000,0.076923,0.600000,1.000000,0.600000
25%,7.200000,4.900000,0.653333,2.400000,2.000000,1.100000
50%,10.800000,7.470000,0.666667,3.600000,3.000000,1.140000
75%,18.000000,12.250000,0.694444,5.700000,5.000000,1.200000
max,46.800000,32.500000,0.800000,15.600000,13.000000,10.000000


In [15]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned dataset: {df.shape}")

Saved cleaned dataset: (10045, 27)
